### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
import textwrap

import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import ComplementNB, MultinomialNB

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [2]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [3]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [4]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [5]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [6]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [7]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [8]:
tfidfvect.vocabulary_.get('cocoliso', -1)

-1

Es muy útil tener el diccionario opuesto que va de índices a términos

In [9]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [10]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [11]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [12]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [13]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [14]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [15]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [16]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [17]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [18]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [19]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [20]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [21]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**

### **1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.


In [22]:
# Fijar semilla para reproducibilidad
np.random.seed(2217)

In [23]:
# Vectorizador por defecto (Punto 1 - como en el template original)
tfidfvect = TfidfVectorizer()

# Matriz documento-término (train) y transformar test
X_train = tfidfvect.fit_transform(newsgroups_train.data)
X_test = tfidfvect.transform(newsgroups_test.data)

# Similitudes test vs train (para Punto 2 - prototipos)
similarities = cosine_similarity(X_test, X_train)

# Info de la matriz
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Vocabulario: {len(tfidfvect.vocabulary_)} términos")
print(f"Similarities shape: {similarities.shape}")

# Diccionarios de vocabulario
vocab = tfidfvect.vocabulary_
idx2word = {i: w for w, i in vocab.items()}

X_train shape: (11314, 101631)
X_test shape: (7532, 101631)
Vocabulario: 101631 términos
Similarities shape: (7532, 11314)


In [24]:

def display_texts(texts, column_width=80, max_rows=25):
    """
    Imprime los textos uno al lado del otro, separados por ' | ' y una línea de guiones.
    Envuelve cada línea al ancho de columna y usa max-rows para no truncar documentos más largos.
    """
    if not texts:
        return
    n = len(texts)
    col = column_width // n
    wrapped = []
    for t in texts:
        w = []
        for line in t:
            if line.strip() == "":
                w.append("")
            else:
                parts = textwrap.wrap(line, width=col) or [""]
                w.extend(parts)
        wrapped.append(w)
    max_rows = min(max_rows, max(len(w) for w in wrapped))
    for i in range(max_rows):
        for w in wrapped:
            s = w[i] if i < len(w) else ""
            if len(s) > col:
                s = s[:col-1] + "…"
            print(s.ljust(col), end=" | ")
        print()
    if max_rows < max(len(w) for w in wrapped):
        for w in wrapped:
            if len(w) > max_rows:
                print("…".ljust(col), end=" | ")
            else:
                print("".ljust(col), end=" | ")
        print()
    print("-" * (col * n + 3 * (n - 1)))

In [25]:
random5 = []
while len(random5) < 5:
    r = np.random.choice(len(newsgroups_test.data))
    txt = [l for l in newsgroups_test.data[r].splitlines() if l.strip()]
    if txt:  # sirve solo si realmente tiene texto
        random5.append(r)
random5 = np.array(random5)

In [26]:
random5_texts = [newsgroups_test.data[r].splitlines() for r in random5]
display_texts(random5_texts, 160, max_rows=50)

                                 | Hi!                              |                                  | Excerpts from                    | In trying to use the Equation    | 
It works for me.  I've run Motif |                                  |                                  | netnews.comp.windows.x:          | editor in Word for Windows 2.0 I | 
1.1.3,1.1.4,1.1.5,1.2,1.2.1, and |         I have a question which  |                                  | 15-May-93 Re: COLORS and X       | get                              | 
1.2.2 on                         | is not directly related to X     |                                  | windows                          | a couple of error messages along | 
an X11R5 server with MotifBC     | Screen Saver.                    |                                  | (A.. John Cwikla@morrison.wri    | the lines of:                    | 
defined.                         |                                  |         Are 180 degree V-6       | (4620)         

In [27]:
similarities = cosine_similarity(X_test, X_train)
top5 = {r: np.argsort(similarities[r])[::-1][0:5] for r in random5}

def display_cmp(r):
    print(f"Documento {r} (clase {newsgroups_test.target_names[y_test[r]]}):")
    for i in top5[r]:
        print(f"  Documento {i} (clase {newsgroups_train.target_names[y_train[i]]})")
    texts = [newsgroups_test.data[r].splitlines()] + [newsgroups_train.data[i].splitlines() for i in top5[r]]
    display_texts(texts, 160, max_rows=40)

In [28]:
display_cmp(random5[0])

Documento 4824 (clase comp.windows.x):
  Documento 367 (clase comp.windows.x)
  Documento 2676 (clase comp.windows.x)
  Documento 2467 (clase comp.windows.x)
  Documento 9733 (clase comp.windows.x)
  Documento 8967 (clase comp.windows.x)
                           |                            | I have a problem with icon | I have a problem with icon | Hi,                        | :                          | 
It works for me.  I've run |   Let me add another of my | pixmap. My application has | pixmap. My application has |                            | : Has anyone found a fix   | 
Motif 1.1.3,1.1.4,1.1.5,1. | concerns: Yes, I can buy a | to run                     | to run                     | I am about to write an     | for the following problem? | 
2,1.2.1, and 1.2.2 on      | port of Motif for "cheap", |    under openwindow and    |    under openwindow and    | application in X/Motif     | :                          | 
an X11R5 server with       | but I cannot get the       | moti

**Doc 4824** (`comp.windows.x`):  
Los 5 vecinos son `comp.windows.x` y discuten problemas de Motif (pixmaps, iconos, OpenWindows vs MWM). Mantienen una misma temática de conversasión y mencionan problemas similares.

In [29]:
display_cmp(random5[1])

Documento 2089 (clase comp.windows.x):
  Documento 3873 (clase comp.windows.x)
  Documento 6057 (clase comp.graphics)
  Documento 8251 (clase comp.graphics)
  Documento 10226 (clase comp.graphics)
  Documento 7133 (clase comp.sys.mac.hardware)
Hi!                        | Environment:               | This seems to be a simple  | At the moment i'm trying   | Hi to all out there.  We   |                            | 
                           |         X11R4              | problem but I just cannot  | to grab a portion of a     | have this problem, and I'm |                            | 
        I have a question  |         Motif 1.1.4        | solve it.                  | Starbase screen, and store | not certain I'm solving it |                            | 
which is not directly      |         Sun IPC 4.1.3      | I wrote a C program to     | it                         | in the correct way.  I was | I think you are suffering  | 
related to X Screen Saver. |                            

**Doc 2089** (`comp.windows.x`):   
**Vecinos:** 1 vecino `comp.windows.x` + 3 `comp.graphics` + 1 `comp.sys.mac.hardware`.  
El documento original pregunta por *X Screen Saver*, los vecinos tratan temas de temas relacionados a la pantalla como capturas, paletas de color, hardware de video. Hay un vocabulario compartido: "screen", "color", "palette". Hay una temática similar (gráficos/pantalla) pero clase distinta.

In [30]:
display_cmp(random5[2])

Documento 1860 (clase rec.autos):
  Documento 5184 (clase rec.autos)
  Documento 5172 (clase sci.space)
  Documento 1001 (clase rec.autos)
  Documento 3908 (clase sci.crypt)
  Documento 8411 (clase comp.graphics)
                           |                            | Let's see. These aren't,   |                            | I think I should also      |                            | 
                           |                            | in a strict sense, amateur | Makes sense, since the new | point out that the         | ...for very small values   | 
                           |                            | rockets. That term         | Mercedes Benz engines go   | mystical DES engines       | of six and nine.           | 
                           | What do you find so wrong  | denotes rockets, the       | from 2.2L-4 to a 2.8L-6.   |         are known          |                            | 
                           | with the flat 6 in the     | engines of which are       | 

**Doc 1860** (`rec.autos`):  
**Vecinos:** 2 `rec.autos` + 1 `sci.space` + 1 `sci.crypt` + 1 `comp.graphics`.  
El documento original pregunta por motores "flat-six", `sci.space` habla de cohetes/propulsión, `sci.crypt` de DES ("engine" como motor criptográfico). Hay ruido por por los usos variados de "engine".

In [31]:
display_cmp(random5[3])

Documento 435 (clase comp.windows.x):
  Documento 1772 (clase comp.graphics)
  Documento 4650 (clase comp.graphics)
  Documento 5451 (clase comp.windows.x)
  Documento 2808 (clase comp.graphics)
  Documento 4234 (clase comp.windows.x)
Excerpts from              | Does anybody out there     |                            | Not being an Xt programmer | You'll probably have to    | Archive-name: Xt-FAQ       | 
netnews.comp.windows.x:    | have or know how to        |                            | by any stretch of the      | set the palette up before  | Version: $Id: FAQ-Xt,v     | 
15-May-93 Re: COLORS and X | calculate the RGB values   | RIX's files with the       | imagination, this is       | you try drawing            | 1.28 93/04/02 12:41:12     | 
windows                    | required to set the 256    | extension  .sci and .scf   | driving me crazy and it's  | in the new colours.        | ware Exp $                 | 
(A.. John                  | color VGA palette so that  | are jus

**Doc 435** (`comp.windows.x`):  
**Vecinos:** 2 vecinos `comp.windows.x` + 3 `comp.graphics`.  
Mantienen los mismos temas referidos a paletas de colores. Hay una frontera difusa entre `comp.windows.x` (X Toolkit) y `comp.graphics` (hardware/paleta de video) ya que `comp.graphics` comparte vocabulario idéntico (paleta, RGB, color, VGA, 256, colourmap). Mas allá de la frontera de clases, los documentos están bien relacionados.

In [32]:
display_cmp(random5[4])

Documento 6352 (clase comp.os.ms-windows.misc):
  Documento 8277 (clase comp.os.ms-windows.misc)
  Documento 2829 (clase comp.os.ms-windows.misc)
  Documento 607 (clase sci.crypt)
  Documento 11278 (clase soc.religion.christian)
  Documento 7496 (clase comp.windows.x)
In trying to use the       |                            |                            | It might be nice to:       | {rest deleted}             | Archive-name:              | 
Equation editor in Word    |         ...                | Adobe has been doing this  |                            |                            | x-faq/speedups             | 
for Windows 2.0 I get      |                            | for years.                 | 1. cut out the ad hominem  | Can the Father possibly    | Last-modified: 1993/4/15   | 
a couple of error messages |         Again, not true.   |                            | attacks on Prof. Denning,  | not hear the words of His  |                            | 
along the lines of:        | Th

**Doc 6352** (`comp.os.ms-windows.misc`):  
**Vecinos:** 2 vecinos `comp.os.ms-windows.misc` + 1 `sci.crypt` + 1 `soc.religion.christian` + 1 `comp.windows.x`.  
`sci.crypt` comparte vocabulario técnico de fuentes/encoding (Type 1, TrueType, Windows ANSI, Mac roman). `comp.windows.x` comparte "font", "Windows". `soc.religion.christian` es ruido por stopwords. De todos los documentos, este es el que tiene menor relación temática con sus vecinos, aunque comparte vocabulario técnico.

**Conclusión:** se logra capturar similitud semántica a nivel temático. Los fallos vienen de:
1. **Vocabulario técnico transversal** ("screen", "color", "palette", "grab", "RGB", "widget" une `comp.windows.x` ↔ `comp.graphics` ↔ `comp.sys.mac.hardware`; "font", "Type 1", "TrueType", "encoding", "Windows" une `comp.os.ms-windows.misc` ↔ `sci.crypt` ↔ `comp.windows.x`)
2. **Polisemia técnica** ("engine" = motor auto / cohete / criptográfico)
3. **Stopwords no filtrados** (`min_df=1`, sin stopwords)

### **2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

In [33]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)
X_test = tfidfvect.transform(newsgroups_test.data)

similarities = cosine_similarity(X_test, X_train) 

# 1-NN: para cada test, el train más similar
y_pred_proto = np.array([
    newsgroups_train.target[np.argmax(similarities[i])]
    for i in range(similarities.shape[0])
])
print(f"pred shape: {y_pred_proto.shape}, ejemplos: {y_pred_proto[:10]}")

pred shape: (7532,), ejemplos: [ 0 19 17 17  0 13 16  6  5  1]


In [34]:
f1_macro = f1_score(newsgroups_test.target, y_pred_proto, average='macro')
print(f"F1 macro (prototipos 1-NN): {f1_macro:.4f}")

print(classification_report(
    newsgroups_test.target, y_pred_proto,
    target_names=newsgroups_test.target_names, zero_division=0
))

F1 macro (prototipos 1-NN): 0.5050
                          precision    recall  f1-score   support

             alt.atheism       0.37      0.51      0.43       319
           comp.graphics       0.54      0.48      0.51       389
 comp.os.ms-windows.misc       0.51      0.46      0.48       394
comp.sys.ibm.pc.hardware       0.52      0.52      0.52       392
   comp.sys.mac.hardware       0.53      0.50      0.52       385
          comp.windows.x       0.70      0.59      0.64       395
            misc.forsale       0.63      0.46      0.53       390
               rec.autos       0.41      0.58      0.48       396
         rec.motorcycles       0.63      0.52      0.57       398
      rec.sport.baseball       0.65      0.54      0.59       397
        rec.sport.hockey       0.75      0.72      0.73       399
               sci.crypt       0.55      0.59      0.57       396
         sci.electronics       0.53      0.33      0.41       393
                 sci.med       0.65     

In [35]:
# Top N términos discriminativos por clase (ratio) -> lista combinada ordenada
N = 50
vocab = tfidfvect.vocabulary_
idx2word = {i: w for w, i in vocab.items()}
n_classes = len(newsgroups_train.target_names)

class_tfidf_sum = np.zeros((n_classes, X_train.shape[1]))
class_counts = np.zeros(n_classes)
for i, c in enumerate(y_train):
    class_tfidf_sum[c] += X_train[i].toarray().ravel()
    class_counts[c] += 1
class_tfidf_mean = class_tfidf_sum / class_counts[:, np.newaxis]

all_terms = []
for c in range(n_classes):
    mean_c = class_tfidf_mean[c]
    mean_others = np.max(np.delete(class_tfidf_mean, c, axis=0), axis=0)
    ratio = mean_c / (mean_others + 1e-10)
    top_idx = np.argsort(ratio)[::-1][:N]
    for i in top_idx:
        all_terms.append({
            'term': idx2word[i],
            'class': newsgroups_train.target_names[c],
            'ratio': ratio[i],
            'mean_in_class': mean_c[i]
        })

# Ordenar por ratio descendente y tomar top N global
all_terms.sort(key=lambda x: x['ratio'], reverse=True)
topN = all_terms[:N]

for t in topN:
    print(f"{t['term']:<15} \t({t['ratio']:.1f}x) \t clase: {t['class']:<25} \t mean: {t['mean_in_class']:.4f}")

armenians       	(218274487.7x) 	 clase: talk.politics.mideast     	 mean: 0.0218
nhl             	(202632638.2x) 	 clase: rec.sport.hockey          	 mean: 0.0203
bikes           	(169793923.2x) 	 clase: rec.motorcycles           	 mean: 0.0170
crypto          	(150869303.7x) 	 clase: sci.crypt                 	 mean: 0.0151
leafs           	(136219167.5x) 	 clase: rec.sport.hockey          	 mean: 0.0136
rsa             	(115300876.4x) 	 clase: sci.crypt                 	 mean: 0.0115
alomar          	(104415379.8x) 	 clase: rec.sport.baseball        	 mean: 0.0104
phillies        	(91832885.9x) 	 clase: rec.sport.baseball        	 mean: 0.0092
hawks           	(89389403.0x) 	 clase: rec.sport.hockey          	 mean: 0.0089
serdar          	(87796109.6x) 	 clase: talk.politics.mideast     	 mean: 0.0088
lciii           	(87317919.1x) 	 clase: comp.sys.mac.hardware     	 mean: 0.0087
bruins          	(84928475.5x) 	 clase: rec.sport.hockey          	 mean: 0.0085
c650            	(844

**F1 macro = 0.5050**, supera al azar (1/20 = 0.05), confirmando que TF-IDF + coseno captura señal temática útil para clasificación.

**Análisis por clase (F1 per-class)**

| Clase | F1 | Términos discriminativos (ratio) |
|-------|-----|----------------------------------|
| `rec.sport.hockey` | **0.73** | nhl, leafs, bruins, jagr, winnipeg, lindros, keenan (> 8Mx) |
| `comp.windows.x` | 0.64 | olwm, xdm (~5Mx) |
| `rec.sport.baseball` | 0.59 | alomar, phillies, hitter, pitcher, clemens, dodgers, baerga, rbi (> 6Mx) |
| `sci.crypt` | 0.57 | crypto, rsa, crypt, decrypt, denning, vesselin (> 6Mx) |
| `rec.motorcycles` | 0.57 | bikes, harley (> 7Mx) |
| `sci.space` | 0.57 | *(no aparece en top 50 global por ratio)* |
| `comp.sys.mac.hardware` | 0.52 | lciii, c650, powerbook, iisi, adb (> 8Mx) |
| `talk.politics.mideast` | 0.45 | armenians, serdar, azerbaijan, gaza, azeri, argic (> 7Mx) |
| `alt.atheism` | 0.43 | bobbe, beauchaine, freewill |
| `talk.politics.guns` | 0.46 | firearm |
| `talk.politics.misc` | 0.31 | genérico, solapa guns/mideast |
| `talk.religion.misc` | **0.28** | rosicrucian |

**Qué revelan los términos discriminativos**

- Los top términos globales por ratio muestran **separación nítida** en clases técnicas/deportivas.
- En cambio, clases **políticas/religiosas** (`talk.*`, `alt.atheism`, `soc.religion.christian`) tienen términos discriminativos más débiles o compartidos, explicando sus F1 bajos (0.28-0.51).

**Limitaciones del 1-NN con vectorizador default**

1. **Gran dimensionalidad**: 101,631 features sparse, un solo vecino ruidoso decide la clase
2. **Sin generalización**: memoriza instancias de train; no aprende distribuciones por clase
3. **Vectorizador default perjudica**: `min_df=1` incluye ruido; `stop_words=None` deja stopwords que diluyen similitud
4. **Fronteras difusas**: `talk.*`, `alt.atheism`, `soc.religion.christian` comparten vocabulario que genera confusión 

### **3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

In [36]:
# Grid search Punto 3 - 12 combos (Clase 1: stop_words, alpha)
vectorizer_grid = [
    {"max_df": 0.9, "min_df": 2, "stop_words": None, "sublinear_tf": True},
    {"max_df": 0.9, "min_df": 2, "stop_words": "english", "sublinear_tf": True},
]

nb_grid = [
    ("MultinomialNB", MultinomialNB, {"alpha": 0.1}),
    ("MultinomialNB", MultinomialNB, {"alpha": 0.5}),
    ("MultinomialNB", MultinomialNB, {"alpha": 1.0}),
    ("ComplementNB", ComplementNB, {"alpha": 0.1}),
    ("ComplementNB", ComplementNB, {"alpha": 0.5}),
    ("ComplementNB", ComplementNB, {"alpha": 1.0}),
]

best_f1 = 0
best_v_params = None
best_clf_name = None
best_clf_params = None

for v_params in vectorizer_grid:
    vec = TfidfVectorizer(**v_params)
    Xtr = vec.fit_transform(newsgroups_train.data)
    Xte = vec.transform(newsgroups_test.data)
    for clf_name, Cls, c_params in nb_grid:
        clf = Cls(**c_params)
        clf.fit(Xtr, newsgroups_train.target)
        pred = clf.predict(Xte)
        f1 = f1_score(newsgroups_test.target, pred, average='macro')
        print(f"vec={v_params} | {clf_name} {c_params} -> F1={f1:.4f}")
        if f1 > best_f1:
            best_f1, best_v_params, best_clf_name, best_clf_params = f1, v_params, clf_name, c_params

print(f"\nBest: F1={best_f1:.4f} | vec={best_v_params} | {best_clf_name} {best_clf_params}")

vec={'max_df': 0.9, 'min_df': 2, 'stop_words': None, 'sublinear_tf': True} | MultinomialNB {'alpha': 0.1} -> F1=0.6716
vec={'max_df': 0.9, 'min_df': 2, 'stop_words': None, 'sublinear_tf': True} | MultinomialNB {'alpha': 0.5} -> F1=0.6318
vec={'max_df': 0.9, 'min_df': 2, 'stop_words': None, 'sublinear_tf': True} | MultinomialNB {'alpha': 1.0} -> F1=0.6016
vec={'max_df': 0.9, 'min_df': 2, 'stop_words': None, 'sublinear_tf': True} | ComplementNB {'alpha': 0.1} -> F1=0.6900
vec={'max_df': 0.9, 'min_df': 2, 'stop_words': None, 'sublinear_tf': True} | ComplementNB {'alpha': 0.5} -> F1=0.6964
vec={'max_df': 0.9, 'min_df': 2, 'stop_words': None, 'sublinear_tf': True} | ComplementNB {'alpha': 1.0} -> F1=0.6932
vec={'max_df': 0.9, 'min_df': 2, 'stop_words': 'english', 'sublinear_tf': True} | MultinomialNB {'alpha': 0.1} -> F1=0.6777
vec={'max_df': 0.9, 'min_df': 2, 'stop_words': 'english', 'sublinear_tf': True} | MultinomialNB {'alpha': 0.5} -> F1=0.6565
vec={'max_df': 0.9, 'min_df': 2, 'stop_wo

In [37]:
# Re-evaluar best model con classification_report completo
best_vec = TfidfVectorizer(**best_v_params)
Xtr_best = best_vec.fit_transform(newsgroups_train.data)
Xte_best = best_vec.transform(newsgroups_test.data)

Clss = {"MultinomialNB": MultinomialNB, "ComplementNB": ComplementNB}
best_clf = Clss[best_clf_name](**best_clf_params)
best_clf.fit(Xtr_best, newsgroups_train.target)
pred_best = best_clf.predict(Xte_best)

print(f"F1 macro best: {f1_score(newsgroups_test.target, pred_best, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(
    newsgroups_test.target, pred_best,
    target_names=newsgroups_test.target_names, zero_division=0
))

F1 macro best: 0.6964

Classification Report:
                          precision    recall  f1-score   support

             alt.atheism       0.32      0.44      0.37       319
           comp.graphics       0.71      0.71      0.71       389
 comp.os.ms-windows.misc       0.72      0.57      0.64       394
comp.sys.ibm.pc.hardware       0.64      0.71      0.67       392
   comp.sys.mac.hardware       0.77      0.73      0.75       385
          comp.windows.x       0.80      0.80      0.80       395
            misc.forsale       0.75      0.74      0.75       390
               rec.autos       0.83      0.74      0.78       396
         rec.motorcycles       0.82      0.78      0.80       398
      rec.sport.baseball       0.91      0.85      0.88       397
        rec.sport.hockey       0.88      0.93      0.90       399
               sci.crypt       0.77      0.80      0.79       396
         sci.electronics       0.71      0.55      0.62       393
                 sci.med     

In [38]:
# Top N por clase desde best_clf.feature_log_prob_ + lista global ordenada
N2 = 50
vocab_best = best_vec.vocabulary_
idx2word_best = {i: w for w, i in vocab_best.items()}

# Lista combinada global (mismo patrón Punto 2, pero con vocab filtrado)
all_terms = []
for c in range(len(newsgroups_train.target_names)):
    logp_c = best_clf.feature_log_prob_[c]
    logp_others = np.max(np.delete(best_clf.feature_log_prob_, c, axis=0), axis=0)
    delta = logp_c - logp_others
    top_idx = np.argsort(delta)[::-1][:N2]
    for i in top_idx:
        all_terms.append({'term': idx2word_best[i], 'class': newsgroups_train.target_names[c], 'delta': delta[i]})
all_terms.sort(key=lambda x: x['delta'], reverse=True)
print("\n=== TOP N GLOBAL DISCRIMINATIVO (delta log_prob) ===")
for t in all_terms[:N2]:
    print(f"{t['term']:<10}\t delta {t['delta']:.2f}\t clase: {t['class']}")


=== TOP N GLOBAL DISCRIMINATIVO (delta log_prob) ===
encryption	 delta 3.45	 clase: sci.crypt
nhl       	 delta 3.24	 clase: rec.sport.hockey
nsa       	 delta 3.24	 clase: sci.crypt
bikes     	 delta 3.10	 clase: rec.motorcycles
armenians 	 delta 3.06	 clase: talk.politics.mideast
geb       	 delta 3.02	 clase: sci.med
arab      	 delta 2.99	 clase: talk.politics.mideast
chastity  	 delta 2.98	 clase: sci.med
n3jxp     	 delta 2.98	 clase: sci.med
dsl       	 delta 2.98	 clase: sci.med
crypto    	 delta 2.93	 clase: sci.crypt
escrow    	 delta 2.92	 clase: sci.crypt
armenian  	 delta 2.88	 clase: talk.politics.mideast
leafs     	 delta 2.84	 clase: rec.sport.hockey
bike      	 delta 2.80	 clase: rec.motorcycles
israeli   	 delta 2.77	 clase: talk.politics.mideast
pitching  	 delta 2.77	 clase: rec.sport.baseball
arabs     	 delta 2.76	 clase: talk.politics.mideast
braves    	 delta 2.73	 clase: rec.sport.baseball
hockey    	 delta 2.72	 clase: rec.sport.hockey
widget    	 delta 2.71	

**Grid:** 2 vectorizadores (stop_words None / english, min_df 2, max_df 0.9, sublinear_tf True) x 6 Naive Bayes (Multinomial / Complement x alpha 0.1 / 0.5 / 1.0).

El grid muestra que ComplementNB supera a MultinomialNB en las seis comparaciones pareadas, lo que es esperable en 20 newsgroups por el desbalance entre clases y la correccion que introduce ComplementNB al estimar probabilidades complementarias. El mejor resultado es **ComplementNB con alpha 0.5 y vectorizador sin stopwords, con F1 0.6964**, claramente por encima del 1-NN del Punto 2 (0.5050). Activar stop_words english no aporta mejora (0.6951) y cambiar alpha tiene efecto distinto segun el modelo: valores chicos (0.1) favorecen a MultinomialNB, mientras que el punto medio (0.5) es optimo para ComplementNB.

**F1 por clase (best model, accuracy 0.72):**

| Clase | F1 | Clase | F1 |
|-------|-----|-------|-----|
| rec.sport.hockey | 0.90 | rec.sport.baseball | 0.88 |
| talk.politics.mideast | 0.82 | sci.med | 0.80 |
| sci.space | 0.80 | rec.motorcycles | 0.80 |
| comp.windows.x | 0.80 | sci.crypt | 0.79 |
| rec.autos | 0.78 | comp.sys.mac.hardware | 0.75 |
| misc.forsale | 0.75 | comp.graphics | 0.71 |
| soc.religion.christian | 0.68 | comp.sys.ibm.pc.hardware | 0.67 |
| talk.politics.guns | 0.66 | comp.os.ms-windows.misc | 0.64 |
| sci.electronics | 0.62 | talk.politics.misc | 0.52 |
| alt.atheism | 0.37 | talk.religion.misc | 0.21 |

Se mantiene la misma jerarquia observada en el Punto 2, pero con una ganancia pareja de entre 0.15 y 0.25 en las clases que ya funcionaban bien. Las categorias con vocabulario mas especifico (deportes, espacio, motos, crypt y windows.x) conservan F1 alto, mientras que las fronteras difusas siguen penalizando a religion.misc (0.21) y atheism (0.37), que comparten gran parte del vocabulario y por eso continuan en la parte baja de la tabla.

**Terminos del best model (delta log_prob, ejemplos):** el clasificador confirma esa lectura. Los pesos mas altos corresponden a entidades muy especificas de cada clase, como hockey con nhl y leafs, crypt con encryption, nsa y crypto, motorcycles con bikes, mideast con armenians y arab, windows.x con widget y space con orbit. Son exactamente los discriminativos que ya aparecian en el Punto 2 y que ahora explican por que esas clases alcanzan el mejor desempenio.

### **4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.

In [39]:
palabras = ["encryption", "intellect", "widget", "orbit", "car"]  

In [40]:
# Matriz termino-documento desde best_vec del Punto 3
X_td_best = Xtr_best.T.tocsr()
vocab_best = best_vec.vocabulary_
idx2word_best = {i: w for w, i in vocab_best.items()}
palabras = [w for w in palabras if w in vocab_best]
print(f"vocab_best {len(vocab_best)}, X_td_best {X_td_best.shape}, palabras {palabras}")

vocab_best 39423, X_td_best (39423, 11314), palabras ['encryption', 'intellect', 'widget', 'orbit', 'car']


In [41]:
for w in palabras:
    idx = vocab_best[w]
    sims = cosine_similarity(X_td_best[idx], X_td_best).ravel()
    top_idx = np.argsort(sims)[::-1][1:6]
    print(f"\n{w}:")
    for j in top_idx:
        print(f"  {idx2word_best[j]:20s} {sims[j]:.4f}")


encryption:
  encrypted            0.3467
  clipper              0.3327
  privacy              0.2927
  secure               0.2704
  scheme               0.2694

intellect:
  n3jxp                0.9849
  chastity             0.9849
  dsl                  0.9840
  skepticism           0.9739
  cadre                0.9666

widget:
  xtwidgettoapplicationcontext 0.3110
  widgets              0.2770
  xtappcontext         0.2662
  xtrealizewidget      0.2412
  propagates           0.2394

orbit:
  hiten                0.3517
  lunar                0.2973
  muc                  0.2811
  redundancy           0.2643
  pluto                0.2622

car:
  cars                 0.1912
  dealer               0.1824
  civic                0.1729
  owner                0.1667
  the                  0.1592


Se transpone `Xtr_best` del Punto 3 (`TfidfVectorizer max_df 0.9, min_df 2, stop_words None, sublinear_tf True`) para obtener `X_td_best` y medir similitud coseno entre filas-palabra. `best_vec` tiene una limpieza con `min_df=2` pero al mantener `stop_words None` aún deja `the`, por eso `car` arrastra ruido.

| Palabra | 5 más similares (coseno) | Lectura |
|---------|---------------------------|---------|
| `encryption` | encrypted 0.34, clipper 0.33, privacy 0.29, secure 0.27, scheme 0.26 | Muy coherente. Todo vocabulario de `sci.crypt` (encriptación, clipper chip, privacidad). Es el caso ideal: término frecuente y específico de la temática crypto. |
| `widget` | xtwidgettoapplicationcontext 0.31, widgets 0.27, xtappcontext 0.26, xtrealizewidget 0.24, propagates 0.23 | Coherente. Familia `comp.windows.x` (Xt Toolkit). Variantes del mismo prefijo `widget/xt` que aparecen en documentación X11. |
| `orbit` | hiten 0.35, lunar 0.29, muc 0.28, redundancy 0.26, pluto 0.26 | Coherente. `sci.space` (misión Hiten, lunar, Pluto). `muc/redundancy` es ruido, pero la temática es espacial. |
| `car` | cars 0.19, dealer 0.18, civic 0.17, owner 0.16, the 0.15 | Parcialmente coherente. `cars/dealer/civic/owner` son semántica de `rec.autos`, pero `the` aparece por usar `stop_words None`. Con `stop_words english` ese vecino desaparecería y entraría otra palabra como podría ser `engine`. |
| `intellect` | n3jxp 0.98, chastity 0.98, dsl 0.98, skepticism 0.97, cadre 0.96 | No coherente. Similitudes cercanas a 1.0 indican que `intellect` y esos 5 términos tienen en común casi exclusivamente un mismo documento de `sci.med`. |

En conjunto, la representación término-documento con TF-IDF captura sinónimos cuando el término es frecuente y temático (`encryption`, `widget`, `orbit`, `car`), pero falla con términos particulares como `intellect` donde la similitud refleja ocurrencia en un único documento y no semántica. El filtrado del Punto 3 mejora respecto al vectorizador default, pero seguiría necesitando stopwords o embeddings para eliminar `the` y capturar sinonimia más allá de co-ocurrencia exacta.